NameError: name 'x_train' is not defined

In [2]:
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

from helpers import *
from implementations import *

In [3]:
import os
import zipfile

DATA_PATH = "data/"

zip_path = os.path.join(DATA_PATH, "dataset_to_release.zip")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_PATH)

print(os.listdir(DATA_PATH))

['dataset', 'dataset_to_release.zip']


In [8]:
"""
Baseline: computes the reference F1 that everyone tries to beat.
"""

import numpy as np
from preprocessing import (
    compute_groups, clean, fit_column_filter, split_train_val,
    fit_preprocessing, apply_preprocessing, train_logistic, sigmoid,
    best_threshold,
)

# ---------------------------------------------------------------------
# Load the data
# Replace this part with the loading code you already use, so that you
# get: x_train, x_test, y_train (labels -1/1) and names (column names).
# ---------------------------------------------------------------------
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("data/dataset")

import csv
import numpy as np

with open("data/dataset/x_train.csv", "r") as f: #ouvrir un fichier csv, le lire et récupérer l'entête
    header = next(csv.reader(f)) #recupere seule l'entête du fichier csv
names = header[1:]   # on enlève "Id" car on ne veut pas l'analyser
assert len(names) == x_train.shape[1]

rows = []
for j, name in enumerate(names):
    col = x_train[:, j]
    valid = col[~np.isnan(col)]
    n_unique = len(np.unique(valid))
    rows.append([
        j, name,
        round(np.isnan(col).mean(), 3),          # la part de cases vides. Python compte True comme 1 et False comme 0, 
        #donc la moyenne donne directement le pourcentage de vides. round(..., 3) arrondit à 3 chiffres après la virgule.
        n_unique,                                 # nombre de valeurs distinctes
        valid.min() if len(valid) else "",
        valid.max() if len(valid) else "",
        round(valid.mean(), 3) if len(valid) else "",#si la colonne a au moins une réponse, calcule ; sinon, laisse la case vide
        ])

# ---------------------------------------------------------------------
# Cleaning (rules learned on x_train, applied identically to x_test)
# ---------------------------------------------------------------------
groups = compute_groups(x_train, names)
x_train_clean, kept_names = clean(x_train, names, groups)
x_test_clean, _ = clean(x_test, names, groups)

keep = fit_column_filter(x_train_clean)
x_train_clean = x_train_clean[:, keep]
x_test_clean = x_test_clean[:, keep]
kept_names = [n for n, k in zip(kept_names, keep) if k]
print("Columns kept:", len(kept_names))

y01 = np.where(y_train == 1, 1, 0)  # -1/1 labels -> 0/1

# ---------------------------------------------------------------------
# Train on 80 %, evaluate on 20 %
# ---------------------------------------------------------------------
i_tr, i_val = split_train_val(len(x_train_clean))

mean, std = fit_preprocessing(x_train_clean[i_tr])
x_tr = apply_preprocessing(x_train_clean[i_tr], mean, std)
x_val = apply_preprocessing(x_train_clean[i_val], mean, std)

w = train_logistic(x_tr, y01[i_tr])

t, score = best_threshold(y01[i_val], sigmoid(x_val @ w))
print(f"\nREFERENCE F1 (validation) = {score:.3f}  with threshold {t:.2f}")


Columns kept: 169
iter 0: loss = 0.6672
iter 100: loss = 0.2664
iter 200: loss = 0.2375
iter 300: loss = 0.2296
iter 400: loss = 0.2263

REFERENCE F1 (validation) = 0.421  with threshold 0.19
